In [1]:
# CELL 1: Check environment
import subprocess
print(subprocess.getoutput('nvidia-smi | grep "GPU 0"'))
import cv2, numpy as np, pandas as pd, os
from tqdm import tqdm
from collections import Counter
print('Libraries loaded.')

/bin/sh: 1: nvidia-smi: not found
Libraries loaded.


In [2]:
# CELL 2: Paths
DATA_ROOT   = '/kaggle/input/datasets/kashmalaamer/avec2014/AVEC2014/AVEC2014'
FUQ_CSV     = '/kaggle/input/datasets/kashmalaamer/avec2014-quantile/avec2014_fuq_results.csv'
GENDER_CSV  = '/kaggle/input/datasets/kashmalaamer/avec2014-gender/avec2014_gender.csv'
WORKING_DIR = '/kaggle/working'
OCC_CSV     = os.path.join(WORKING_DIR, 'avec2014_occlusion.csv')

print(f'DATA_ROOT  exists: {os.path.exists(DATA_ROOT)}')
print(f'FUQ_CSV    exists: {os.path.exists(FUQ_CSV)}')
print(f'GENDER_CSV exists: {os.path.exists(GENDER_CSV)}')

DATA_ROOT  exists: True
FUQ_CSV    exists: True
GENDER_CSV exists: True


In [3]:
# CELL 3: Collect all video paths
def collect_all_videos(data_root):
    items = []
    for split in ['Training', 'Development', 'Testing']:
        for task in ['Freeform', 'Northwind']:
            folder = os.path.join(data_root, split, task)
            if not os.path.exists(folder):
                continue
            for f in sorted(os.listdir(folder)):
                if not f.lower().endswith('.mp4'):
                    continue
                path = os.path.join(folder, f)
                rel  = os.path.relpath(path, data_root).replace('\\', '/')
                stem = os.path.splitext(rel)[0]
                items.append((path, stem))
    return items

all_videos = collect_all_videos(DATA_ROOT)
print(f'Total videos: {len(all_videos)}')
print(f'Sample: {all_videos[0][1]}')

Total videos: 303
Sample: Training/Freeform/203_1_Freeform_video


In [6]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def detect_glasses(face_roi):
    h, w = face_roi.shape[:2]
    eye_region = face_roi[int(h*0.2):int(h*0.5), :]
    if eye_region.size == 0:
        return 0
    gray    = cv2.cvtColor(eye_region, cv2.COLOR_BGR2GRAY)
    edges   = cv2.Canny(gray, 50, 150)
    density = np.sum(edges > 0) / edges.size
    return 1 if density > 0.12 else 0

def detect_beard(face_roi):
    h, w = face_roi.shape[:2]
    lower = face_roi[int(h*0.6):, :]
    if lower.size == 0:
        return 0
    hsv        = cv2.cvtColor(lower, cv2.COLOR_BGR2HSV)
    dark_mask  = (hsv[:,:,2] < 100) & (hsv[:,:,1] < 80)
    dark_ratio = np.sum(dark_mask) / dark_mask.size
    return 1 if dark_ratio > 0.25 else 0

def detect_occlusion_video(path, sample_every=30):
    cap     = cv2.VideoCapture(path)
    glasses = []
    beards  = []
    idx     = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % sample_every == 0:
            gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(
                gray, scaleFactor=1.1, minNeighbors=4, minSize=(60, 60)
            )
            if len(faces) > 0:
                x, y, w, h = max(faces, key=lambda f: f[2]*f[3])
                face_roi   = frame[y:y+h, x:x+w]
                if face_roi.size > 0:
                    glasses.append(detect_glasses(face_roi))
                    beards.append(detect_beard(face_roi))
        idx += 1
    cap.release()
    if not glasses:
        return 0, 0
    has_glasses = 1 if sum(glasses)/len(glasses) > 0.3 else 0
    has_beard   = 1 if sum(beards)/len(beards) > 0.3 else 0
    return has_glasses, has_beard

print('Functions defined.')
print(f'Face cascade loaded: {face_cascade is not None}')

Functions defined.
Face cascade loaded: True


In [7]:
# CELL 5: Run occlusion detection on all videos (~15-20 min)
print(f'Processing {len(all_videos)} videos...')
rows = []
no_face_count = 0

for path, stem in tqdm(all_videos):
    try:
        glasses, beard = detect_occlusion_video(path)
        rows.append({
            'filename':    stem,
            'has_glasses': glasses,
            'has_beard':   beard
        })
    except Exception as e:
        rows.append({'filename': stem, 'has_glasses': 0, 'has_beard': 0})
        no_face_count += 1

occ_df = pd.DataFrame(rows)
print(f'\nDone. Videos with no face detected: {no_face_count}')
print(f'\nGlasses distribution:')
print(occ_df['has_glasses'].value_counts().rename({0:'No Glasses', 1:'Glasses'}))
print(f'\nBeard distribution:')
print(occ_df['has_beard'].value_counts().rename({0:'No Beard', 1:'Beard'}))

occ_df.to_csv(OCC_CSV, index=False)
print(f'\nSaved: {OCC_CSV}')

Processing 303 videos...


100%|██████████| 303/303 [15:59<00:00,  3.17s/it]


Done. Videos with no face detected: 0

Glasses distribution:
has_glasses
No Glasses    265
Glasses        38
Name: count, dtype: int64

Beard distribution:
has_beard
No Beard    291
Beard        12
Name: count, dtype: int64

Saved: /kaggle/working/avec2014_occlusion.csv


In [8]:
# CELL 6: Load FUQ results and merge with occlusion annotations
test_fuq_df = pd.read_csv(FUQ_CSV)
gender_df   = pd.read_csv(GENDER_CSV)

# Build maps
occ_map = occ_df.set_index('filename')[['has_glasses', 'has_beard']].to_dict('index')

test_fuq_df['has_glasses'] = test_fuq_df['stem'].map(
    lambda s: occ_map.get(s, {}).get('has_glasses', 0)
)
test_fuq_df['has_beard'] = test_fuq_df['stem'].map(
    lambda s: occ_map.get(s, {}).get('has_beard', 0)
)

print(f'Test set: {len(test_fuq_df)} videos')
print(f'Glasses in test: {test_fuq_df["has_glasses"].sum()}')
print(f'Beards in test:  {test_fuq_df["has_beard"].sum()}')
print(f'\nTest fuq columns: {test_fuq_df.columns.tolist()}')
print(test_fuq_df[['stem','gender','has_glasses','has_beard','covered']].head(10))

Test set: 50 videos
Glasses in test: 8
Beards in test:  2

Test fuq columns: ['stem', 'y_true', 'y_pred', 'lo', 'hi', 'gender', 'covered', 'has_glasses', 'has_beard']
                                    stem gender  has_glasses  has_beard  \
0  Testing/Freeform/203_2_Freeform_video      M            0          0   
1  Testing/Freeform/206_2_Freeform_video      M            1          0   
2  Testing/Freeform/210_2_Freeform_video      M            0          0   
3  Testing/Freeform/211_2_Freeform_video      M            1          0   
4  Testing/Freeform/212_1_Freeform_video      M            0          0   
5  Testing/Freeform/214_3_Freeform_video      M            0          0   
6  Testing/Freeform/218_3_Freeform_video      M            0          0   
7  Testing/Freeform/220_1_Freeform_video      M            0          0   
8  Testing/Freeform/220_3_Freeform_video      F            0          0   
9  Testing/Freeform/224_1_Freeform_video      M            0          0   

   cove

In [10]:
# CELL 7: Occlusion FUQ analysis
print('=== FUQ Results by Glasses ===')
print(f'{"Group":20s} {"N":>5s} {"PICP":>8s} {"MPIW":>8s} {"MAE":>8s}')
print('-' * 55)
glasses_picps = {}
for val, lbl in [(0, 'No Glasses'), (1, 'Glasses')]:
    grp = test_fuq_df[test_fuq_df['has_glasses'] == val]
    if len(grp) == 0:
        print(f'{lbl:20s}     0  (none detected)')
        continue
    picp = grp['covered'].mean()
    mpiw = (grp['hi'] - grp['lo']).mean()
    mae  = (grp['y_true'] - grp['y_pred']).abs().mean()
    glasses_picps[lbl] = picp
    print(f'{lbl:20s} {len(grp):>5d} {picp:>8.4f} {mpiw:>8.4f} {mae:>8.4f}')

print('\n=== FUQ Results by Beard ===')
print(f'{"Group":20s} {"N":>5s} {"PICP":>8s} {"MPIW":>8s} {"MAE":>8s}')
print('-' * 55)
beard_picps = {}
for val, lbl in [(0, 'No Beard'), (1, 'Beard')]:
    grp = test_fuq_df[test_fuq_df['has_beard'] == val]
    if len(grp) == 0:
        print(f'{lbl:20s}     0  (none detected)')
        continue
    picp = grp['covered'].mean()
    mpiw = (grp['hi'] - grp['lo']).mean()
    mae  = (grp['y_true'] - grp['y_pred']).abs().mean()
    beard_picps[lbl] = picp
    print(f'{lbl:20s} {len(grp):>5d} {picp:>8.4f} {mpiw:>8.4f} {mae:>8.4f}')

# Gaps
glasses_gap = abs(
    glasses_picps.get('No Glasses', 0) - glasses_picps.get('Glasses', 0)
)
beard_gap = abs(
    beard_picps.get('No Beard', 0) - beard_picps.get('Beard', 0)
)
print(f'\nGlasses PICP Gap : {glasses_gap:.4f}')
print(f'Beard   PICP Gap : {beard_gap:.4f}')

=== FUQ Results by Glasses ===
Group                    N     PICP     MPIW      MAE
-------------------------------------------------------
No Glasses              42   0.9048  29.8614   8.0041
Glasses                  8   0.7500  31.9503  10.6655

=== FUQ Results by Beard ===
Group                    N     PICP     MPIW      MAE
-------------------------------------------------------
No Beard                48   0.8958  29.9756   8.3347
Beard                    2   0.5000  35.4772  10.7150

Glasses PICP Gap : 0.1548
Beard   PICP Gap : 0.3958


In [11]:
# CELL 8: Gender x Occlusion interaction
# Does occlusion affect men and women differently?

print('=== Beard x Gender interaction ===')
print(f'{"Group":25s} {"N":>5s} {"PICP":>8s} {"MAE":>8s}')
print('-' * 50)
for beard_val, beard_lbl in [(0, 'No Beard'), (1, 'Beard')]:
    for g, g_lbl in [('M', 'Male'), ('F', 'Female')]:
        grp = test_fuq_df[
            (test_fuq_df['has_beard'] == beard_val) &
            (test_fuq_df['gender'] == g)
        ]
        if len(grp) == 0:
            continue
        lbl  = f'{beard_lbl} {g_lbl}'
        picp = grp['covered'].mean()
        mae  = (grp['y_true'] - grp['y_pred']).abs().mean()
        print(f'{lbl:25s} {len(grp):>5d} {picp:>8.4f} {mae:>8.4f}')

print('\n=== Glasses x Gender interaction ===')
print(f'{"Group":25s} {"N":>5s} {"PICP":>8s} {"MAE":>8s}')
print('-' * 50)
for g_val, g_lbl in [(0, 'No Glasses'), (1, 'Glasses')]:
    for g, g_lbl2 in [('M', 'Male'), ('F', 'Female')]:
        grp = test_fuq_df[
            (test_fuq_df['has_glasses'] == g_val) &
            (test_fuq_df['gender'] == g)
        ]
        if len(grp) == 0:
            continue
        lbl  = f'{g_lbl} {g_lbl2}'
        picp = grp['covered'].mean()
        mae  = (grp['y_true'] - grp['y_pred']).abs().mean()
        print(f'{lbl:25s} {len(grp):>5d} {picp:>8.4f} {mae:>8.4f}')

=== Beard x Gender interaction ===
Group                         N     PICP      MAE
--------------------------------------------------
No Beard Male                34   0.9118   8.2101
No Beard Female              14   0.8571   8.6372
Beard Male                    1   0.0000  21.0183
Beard Female                  1   1.0000   0.4117

=== Glasses x Gender interaction ===
Group                         N     PICP      MAE
--------------------------------------------------
No Glasses Male              28   0.8929   8.3438
No Glasses Female            14   0.9286   7.3247
Glasses Male                  7   0.8571   9.5053
Glasses Female                1   0.0000  18.7868


In [12]:
# CELL 9: Save occlusion results
# Also save merged test results with occlusion flags

merged_csv = os.path.join(WORKING_DIR, 'avec2014_fuq_with_occlusion.csv')
test_fuq_df.to_csv(merged_csv, index=False)
print(f'Merged results saved: {merged_csv}')
print(f'Occlusion CSV saved:  {OCC_CSV}')

print('\n===== Occlusion Summary =====')
print(f'Total test videos : {len(test_fuq_df)}')
print(f'With glasses      : {test_fuq_df["has_glasses"].sum()}')
print(f'With beard        : {test_fuq_df["has_beard"].sum()}')
print(f'Glasses PICP Gap  : {glasses_gap:.4f}')
print(f'Beard   PICP Gap  : {beard_gap:.4f}')
print('\nDownload results from the Output tab.')

Merged results saved: /kaggle/working/avec2014_fuq_with_occlusion.csv
Occlusion CSV saved:  /kaggle/working/avec2014_occlusion.csv

===== Occlusion Summary =====
Total test videos : 50
With glasses      : 8
With beard        : 2
Glasses PICP Gap  : 0.1548
Beard   PICP Gap  : 0.3958

Download results from the Output tab.
